# BUSI · разведочный анализ
Выполните ячейки из копии репозитория после `python -m busi.prepare`. В Colab укажите папку репозитория через BUSI_PROJECT и подключите Google Drive. Переменная BUSI_ROOT переопределяет путь к изображениям.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

project = Path(os.environ.get("BUSI_PROJECT", ".")).resolve()
if project.name == "notebooks":
    project = project.parent
art = project / "artifacts"
root = Path(os.environ.get("BUSI_ROOT", project / "data/busi"))
if not root.exists() and Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/colab/data/busi")
out = art / "eda"
out.mkdir(exist_ok=True)
df = pd.read_csv(art / "manifest.csv")
audit = pd.read_csv(art / "audit.csv")
classes = ["normal", "benign", "malignant"]
counts = df["class"].value_counts().reindex(classes)
display(pd.DataFrame({"Снимки": counts, "Доля": counts / counts.sum()}))
def save(fig, name):
    fig.tight_layout()
    fig.savefig(out / name, dpi=125)
    plt.show()
    plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
counts.plot.bar(ax=axes[0], title="Количество снимков", rot=0)
(counts / counts.sum() * 100).plot.bar(ax=axes[1], title="Доля снимков, %", rot=0)
save(fig, "classes.png")
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
df[["width","height"]].plot.hist(ax=axes[0], alpha=.5, bins=25, title="Размеры, пиксели")
df.channels.value_counts().sort_index().plot.bar(ax=axes[1], title="Число каналов", rot=0)
save(fig, "dimensions.png")

In [ ]:
def gallery(rows, name):
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 3), squeeze=False)
    for ax, row in zip(axes.flat, rows.itertuples()):
        with Image.open(root / row.filepath) as im:
            ax.imshow(im.convert("L"), cmap="gray", vmin=0, vmax=255)
        ax.set_title(Path(row.filepath).name)
        ax.axis("off")
    save(fig, name)

gallery(df.groupby("class").sample(1, random_state=42), "examples.png")
review = df.loc[[df.width.idxmin(), df.width.idxmax(), df.height.idxmin(), df.height.idxmax()]]
gallery(review.drop_duplicates("filepath"), "quality.png")
display(audit.issue.value_counts())
display(Markdown(f"Самый частый класс **{counts.idxmax()}** встречается в {counts.max()/counts.min():.2f} раза чаще самого редкого **{counts.idxmin()}**. Для сравнения моделей используем macro-F1 и показатели каждого класса."))